In [1]:
# Fine tune the gpt-3 model
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [2]:
import datasets
from datasets import load_dataset_builder, DatasetBuilder, list_datasets, load_dataset

datasets_list = list_datasets()

C:\Users\burha\AppData\Local\Temp\ipykernel_23800\1135615729.py:4: FutureWarning: list_datasets is deprecated and will be removed in the next major version of datasets. Use 'huggingface_hub.list_datasets' instead.
  datasets_list = list_datasets()


In [3]:
data = load_dataset(
    "json",
    data_files="./train.jsonl",
    split='train'
)

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
data

Dataset({
    features: ['messages'],
    num_rows: 14
})

In [5]:
import openai

In [11]:
res = openai.File.create(
    file=open("train.jsonl", "r"),
    purpose="fine-tune"
)

In [12]:
res

<File file id=file-Oni0aRW0zgCxDzuOyaTBvVRr at 0x1cd76ade510> JSON: {
  "object": "file",
  "id": "file-Oni0aRW0zgCxDzuOyaTBvVRr",
  "purpose": "fine-tune",
  "filename": "file",
  "bytes": 5675,
  "created_at": 1697095646,
  "status": "uploaded",
  "status_details": null
}

In [13]:
file_id = res['id']
file_id

'file-Oni0aRW0zgCxDzuOyaTBvVRr'

In [14]:
res = openai.FineTuningJob.create(
    training_file=file_id,
    model='gpt-3.5-turbo'
)
res

<FineTuningJob fine_tuning.job id=ftjob-SkQpYGfN71JmNxL5kPUWNLBT at 0x1cd5bbee2d0> JSON: {
  "object": "fine_tuning.job",
  "id": "ftjob-SkQpYGfN71JmNxL5kPUWNLBT",
  "model": "gpt-3.5-turbo-0613",
  "created_at": 1697095716,
  "finished_at": null,
  "fine_tuned_model": null,
  "organization_id": "org-a2H3bfzl3yjUkgCKlJyxDunZ",
  "result_files": [],
  "status": "validating_files",
  "validation_file": null,
  "training_file": "file-Oni0aRW0zgCxDzuOyaTBvVRr",
  "hyperparameters": {
    "n_epochs": "auto"
  },
  "trained_tokens": null,
  "error": null
}

In [15]:
job_id = res['id']
job_id

'ftjob-SkQpYGfN71JmNxL5kPUWNLBT'

In [17]:
from time import sleep

while True:
    res = openai.FineTuningJob.retrieve(job_id)
    if res['finished_at'] != None:
        break
    else:
        print(".", end="")
        sleep(100)

...

In [18]:
res

<FineTuningJob fine_tuning.job id=ftjob-SkQpYGfN71JmNxL5kPUWNLBT at 0x1cd76afb2f0> JSON: {
  "object": "fine_tuning.job",
  "id": "ftjob-SkQpYGfN71JmNxL5kPUWNLBT",
  "model": "gpt-3.5-turbo-0613",
  "created_at": 1697095716,
  "finished_at": 1697096060,
  "fine_tuned_model": "ft:gpt-3.5-turbo-0613:mentorskool::88kV3pt8",
  "organization_id": "org-a2H3bfzl3yjUkgCKlJyxDunZ",
  "result_files": [
    "file-iUAcNNLrWjr9zrsrbrQ0lnPu"
  ],
  "status": "succeeded",
  "validation_file": null,
  "training_file": "file-Oni0aRW0zgCxDzuOyaTBvVRr",
  "hyperparameters": {
    "n_epochs": 7
  },
  "trained_tokens": 7035,
  "error": null
}

In [19]:
# fetch out the model
fine_tune_model = res['fine_tuned_model']
fine_tune_model

'ft:gpt-3.5-turbo-0613:mentorskool::88kV3pt8'

In [20]:
from langchain.chat_models import ChatOpenAI

In [21]:
llm = ChatOpenAI(
    model_name=fine_tune_model,
    temperature=0
)

In [23]:
llm.predict("Provide a sql query for how many clients are there in the database?")

'SELECT COUNT(client_id) FROM clients;'

In [25]:
llm.predict("Provide a sql query on how can we find valid email_id of clients?")

"SELECT email_id FROM clients WHERE email_id LIKE '%@%.%';"

In [26]:
llm.predict("Which is the primary key of progress_fact table?")

'The primary key of the progress_fact table is not specified in the given information. The primary key is typically a unique identifier for each row in a table, and it is usually chosen based on the specific requirements of the database design.'

In [34]:
llm.predict("Provide a sql query that gives the top 5 learners from skills_fact who scored highest in Data quality essentials")

"SELECT learner_fact_id, score\nFROM skills_fact\nWHERE skill_name = 'Data quality essentials'\nORDER BY score DESC\nLIMIT 5;"

In [29]:
message = "Which is the primary key of progress_fact table?"
answer = openai.ChatCompletion.create(
    model=fine_tune_model,
    messages=[{"role": "user", "content": message}],
    temperature=0,
)

In [30]:
answer

<OpenAIObject chat.completion id=chatcmpl-88kisAaMq1ZwBGJc965f7sDIA1yte at 0x1cd0f951190> JSON: {
  "id": "chatcmpl-88kisAaMq1ZwBGJc965f7sDIA1yte",
  "object": "chat.completion",
  "created": 1697096918,
  "model": "ft:gpt-3.5-turbo-0613:mentorskool::88kV3pt8",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "The primary key of the progress_fact table is not specified in the given information. The primary key is typically a unique identifier for each row in a table, and it is usually chosen based on the specific requirements of the database design."
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 17,
    "completion_tokens": 46,
    "total_tokens": 63
  }
}

In [31]:
answer.choices[0]["message"]["content"]

'The primary key of the progress_fact table is not specified in the given information. The primary key is typically a unique identifier for each row in a table, and it is usually chosen based on the specific requirements of the database design.'

In [36]:
## Mistake occured while fine-tuning, explore more

# Have added some more questions, now again try to fine tune the model

In [1]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [2]:
import openai
import json

In [4]:
upload_response = openai.File.create(
    file=open('train.jsonl', 'rb'),
    purpose="fine-tune"
)

In [5]:
upload_response

<File file id=file-xv71Utqnub5Q8mpQuQ9Wa2Uc at 0x20f5d4d6690> JSON: {
  "object": "file",
  "id": "file-xv71Utqnub5Q8mpQuQ9Wa2Uc",
  "purpose": "fine-tune",
  "filename": "file",
  "bytes": 38674,
  "created_at": 1697113730,
  "status": "uploaded",
  "status_details": null
}

In [6]:
file_id = upload_response['id']

In [7]:
# Now let's create the model
response = openai.FineTuningJob.create(
    training_file=file_id,
    model='gpt-3.5-turbo'
)
response

<FineTuningJob fine_tuning.job id=ftjob-XiUEkWmPLYl9QHo3xk4GZvN3 at 0x20f5b92c8f0> JSON: {
  "object": "fine_tuning.job",
  "id": "ftjob-XiUEkWmPLYl9QHo3xk4GZvN3",
  "model": "gpt-3.5-turbo-0613",
  "created_at": 1697113851,
  "finished_at": null,
  "fine_tuned_model": null,
  "organization_id": "org-a2H3bfzl3yjUkgCKlJyxDunZ",
  "result_files": [],
  "status": "validating_files",
  "validation_file": null,
  "training_file": "file-xv71Utqnub5Q8mpQuQ9Wa2Uc",
  "hyperparameters": {
    "n_epochs": "auto"
  },
  "trained_tokens": null,
  "error": null
}

In [8]:
# fetch the job_id
job_id = response['id']
job_id

'ftjob-XiUEkWmPLYl9QHo3xk4GZvN3'

In [9]:
# See whether model is trained or not
openai.FineTuningJob.retrieve(job_id)

<FineTuningJob fine_tuning.job id=ftjob-XiUEkWmPLYl9QHo3xk4GZvN3 at 0x20f5935f650> JSON: {
  "object": "fine_tuning.job",
  "id": "ftjob-XiUEkWmPLYl9QHo3xk4GZvN3",
  "model": "gpt-3.5-turbo-0613",
  "created_at": 1697113851,
  "finished_at": 1697114147,
  "fine_tuned_model": "ft:gpt-3.5-turbo-0613:mentorskool::88pClEW6",
  "organization_id": "org-a2H3bfzl3yjUkgCKlJyxDunZ",
  "result_files": [
    "file-aplsrwaDlLG8XTs2AhCKM6m8"
  ],
  "status": "succeeded",
  "validation_file": null,
  "training_file": "file-xv71Utqnub5Q8mpQuQ9Wa2Uc",
  "hyperparameters": {
    "n_epochs": 4
  },
  "trained_tokens": 30916,
  "error": null
}

In [10]:
res = openai.FineTuningJob.retrieve(job_id)
res

<FineTuningJob fine_tuning.job id=ftjob-XiUEkWmPLYl9QHo3xk4GZvN3 at 0x20f5d4769f0> JSON: {
  "object": "fine_tuning.job",
  "id": "ftjob-XiUEkWmPLYl9QHo3xk4GZvN3",
  "model": "gpt-3.5-turbo-0613",
  "created_at": 1697113851,
  "finished_at": 1697114147,
  "fine_tuned_model": "ft:gpt-3.5-turbo-0613:mentorskool::88pClEW6",
  "organization_id": "org-a2H3bfzl3yjUkgCKlJyxDunZ",
  "result_files": [
    "file-aplsrwaDlLG8XTs2AhCKM6m8"
  ],
  "status": "succeeded",
  "validation_file": null,
  "training_file": "file-xv71Utqnub5Q8mpQuQ9Wa2Uc",
  "hyperparameters": {
    "n_epochs": 4
  },
  "trained_tokens": 30916,
  "error": null
}

In [11]:
# Model is trained, so fetch the model_name
ft_model = res['fine_tuned_model'] # "ft:gpt-3.5-turbo-0613:mentorskool::88pClEW6"
ft_model

'ft:gpt-3.5-turbo-0613:mentorskool::88pClEW6'

In [15]:
# Now the model is trained so let's check whether it is properly fine-tuned or not
message = "What can be the sql query when we have to filter the top 5 participants based on their total scores in the Excel Assessment"
answer = openai.ChatCompletion.create(
    model=ft_model,
    messages=[{"role": "user", "content": message}],
    temperature=0,
)

In [16]:
answer

<OpenAIObject chat.completion id=chatcmpl-88pI4WoConVj5uxO0zejFQhUDbh5k at 0x20f5b30a5d0> JSON: {
  "id": "chatcmpl-88pI4WoConVj5uxO0zejFQhUDbh5k",
  "object": "chat.completion",
  "created": 1697114476,
  "model": "ft:gpt-3.5-turbo-0613:mentorskool::88pClEW6",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Assuming you have a table named \"Participants\" with columns \"Name\" and \"Score\" in your database, the SQL query to filter the top 5 participants based on their total scores in the Excel Assessment would be:\n\nSELECT Name, SUM(Score) AS TotalScore\nFROM Participants\nGROUP BY Name\nORDER BY TotalScore DESC\nLIMIT 5;"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 32,
    "completion_tokens": 73,
    "total_tokens": 105
  }
}

In [68]:
answer = openai.ChatCompletion.create(
  model=ft_model,
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
  ]
)

In [69]:
print(answer["choices"][0]["message"])

{
  "role": "assistant",
  "content": "Hi there! How can I assist you today?"
}


In [72]:
message = "Provide a sql query where we can find that How many clients have valid email ids?"
answer = openai.ChatCompletion.create(
  model=ft_model,
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": message}
  ]
)

In [73]:
print(answer["choices"][0]["message"])

{
  "role": "assistant",
  "content": "Sure! Here's a SQL query that counts the number of clients with valid email IDs:\n\n```sql\nSELECT COUNT(client_id) AS num_clients_valid_email \nFROM clients \nWHERE email LIKE '%_@__%.__%' \n```\n\nThis query selects the client_id column from the clients table and applies a condition in the WHERE clause to only consider email IDs that have at least one character before the \"@\" symbol, followed by at least one character, then a \".\", and finally at least one character after the \".\" symbol. The result is the count of clients with valid email IDs."
}


In [74]:
message = "When user asks to find the scores in percentage then which table should be referred?"
answer = openai.ChatCompletion.create(
  model=ft_model,
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": message}
  ]
)

In [75]:
print(answer["choices"][0]["message"])

{
  "role": "assistant",
  "content": "To find the scores in percentage, the table that should be referred is the one that contains the raw scores and total marks obtained by each individual or student."
}


> Above we can see that only fine tuning won't work so we have to apply the embedding also, So let's perform embedding on the fine tuned model

In [28]:
from langchain.document_loaders import PyPDFLoader
from langchain.llms import OpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import DeepLake
from langchain.text_splitter import CharacterTextSplitter

In [30]:
# Load the pdf
loader = PyPDFLoader("Enqurious_ETL_DB document.pdf")

In [31]:
# Now split the pdf into pages
pages = loader.load_and_split()

In [32]:
len(pages)

36

In [33]:
# Now split the documents via CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

In [34]:
docs = text_splitter.split_documents(pages)

In [35]:
len(docs)

36

In [36]:
# Now create an embedding object
embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')

In [ ]:
import os

In [37]:
my_activeloop_org_id = os.environ.get("my_activeloop_org_id")
my_activeloop_datset_name = os.environ.get("my_activeloop_datset_name")
dataset_path = f"hub://{my_activeloop_org_id}/{my_activeloop_datset_name}"

In [38]:
db = DeepLake(dataset_path=dataset_path, embedding_function=embeddings)

Using embedding function is deprecated and will be removed in the future. Please use embedding instead.


Deep Lake Dataset in hub://burhanuddinnahargarwala/enqurious_practice already exists, loading from the storage


In [39]:
# Now add documents to the vector db
db.add_documents(docs)

100%|██████████| 36/36 [00:06<00:00,  5.72it/s]
/

Dataset(path='hub://burhanuddinnahargarwala/enqurious_practice', tensors=['embedding', 'id', 'metadata', 'text'])

  tensor      htype      shape      dtype  compression
  -------    -------    -------    -------  ------- 
 embedding  embedding  (72, 1536)  float32   None   
    id        text      (72, 1)      str     None   
 metadata     json      (72, 1)      str     None   
   text       text      (72, 1)      str     None   


['9f390953-68fe-11ee-b60b-d9950286f01c',
 '9f390954-68fe-11ee-98a7-d9950286f01c',
 '9f390955-68fe-11ee-b082-d9950286f01c',
 '9f390956-68fe-11ee-b8f9-d9950286f01c',
 '9f390957-68fe-11ee-9aea-d9950286f01c',
 '9f390958-68fe-11ee-8297-d9950286f01c',
 '9f390959-68fe-11ee-9092-d9950286f01c',
 '9f39095a-68fe-11ee-bdca-d9950286f01c',
 '9f39095b-68fe-11ee-a0b8-d9950286f01c',
 '9f39095c-68fe-11ee-9f19-d9950286f01c',
 '9f39095d-68fe-11ee-99ea-d9950286f01c',
 '9f39095e-68fe-11ee-bac4-d9950286f01c',
 '9f39095f-68fe-11ee-b229-d9950286f01c',
 '9f390960-68fe-11ee-b51c-d9950286f01c',
 '9f390961-68fe-11ee-9369-d9950286f01c',
 '9f390962-68fe-11ee-bdf0-d9950286f01c',
 '9f390963-68fe-11ee-9e80-d9950286f01c',
 '9f390964-68fe-11ee-96ea-d9950286f01c',
 '9f390965-68fe-11ee-a556-d9950286f01c',
 '9f390966-68fe-11ee-b910-d9950286f01c',
 '9f390967-68fe-11ee-ba1e-d9950286f01c',
 '9f390968-68fe-11ee-8dfa-d9950286f01c',
 '9f390969-68fe-11ee-a04c-d9950286f01c',
 '9f39096a-68fe-11ee-88e9-d9950286f01c',
 '9f39096b-68fe-

In [40]:
# Now fetch the retrievr from the db
retriever = db.as_retriever()

In [41]:
from langchain.chains import RetrievalQA

In [52]:
from langchain.chat_models import ChatOpenAI

In [53]:
llm = ChatOpenAI(model_name=ft_model, temperature=0)

In [54]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

In [55]:
from langchain.tools import Tool

In [56]:
tools = [
    Tool(
        name="Enqurious ETL Document",
        description="Contains the data dictionary of tables present in the Enqurious ETL database. Also there are explanation of different ETL processes such as ETL Extraction process, ETL transformation process, ETL loading process.",
        func=qa_chain.run
    )
]

In [57]:
from langchain.agents import initialize_agent

In [58]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    verbose=True
)

In [59]:
agent.run("When user ask the question related to the progress of learners then which table should be referred?")



> Entering new AgentExecutor chain...
I need to find the table that contains information about the progress of learners.
Action: Enqurious ETL Document
Action Input: Search for tables related to learner progress.
Observation: The tables related to learner progress are "progress_fact" and "skill_fact".
Thought:The tables "progress_fact" and "skill_fact" contain information about the progress of learners.
Final Answer: The tables "progress_fact" and "skill_fact" should be referred when a user asks a question related to the progress of learners.

> Finished chain.


'The tables "progress_fact" and "skill_fact" should be referred when a user asks a question related to the progress of learners.'

In [62]:
agent.run("How to filter the top 5 participants based on their total scores in the Excel Essntials project from the skills_fact table?")



> Entering new AgentExecutor chain...
I need to find the table that contains the total scores of participants in the Excel Essentials project.
Action: Enqurious ETL Document
Action Input: Search for the table that contains the total scores of participants in the Excel Essentials project.
Observation: Based on the given context, there is no specific mention of a table that contains the total scores of participants in the Excel Essentials project. Therefore, I don't have the information to answer your question.
Thought:I don't have enough information to answer the question.
Final Answer: I don't have enough information to answer the question.

> Finished chain.


"I don't have enough information to answer the question."

In [63]:
agent.run("Will you provide me an overview on your context?")



> Entering new AgentExecutor chain...
I need to understand the context of the Enqurious ETL database.
Action: Enqurious ETL Document
Action Input: None
Observation: I don't know.
Thought:I need to look for an overview section in the Enqurious ETL Document.
Action: Enqurious ETL Document
Action Input: Overview
Observation: The ETL documentation provides a detailed overview of the ETL (Extract, Transform, Load) process for Enqurious 1.2. It includes information on the purpose and objectives of the ETL process, as well as details about the data sources, extraction methods, transformation steps, data loading mechanism, error handling, and schedule or dependencies of the ETL process.
Thought:I now know the context of the Enqurious ETL database.
Final Answer: The Enqurious ETL database is used for the ETL (Extract, Transform, Load) process in Enqurious 1.2. It includes information on data sources, extraction methods, transformation steps, data loading mechanism, error handling, and schedul

'The Enqurious ETL database is used for the ETL (Extract, Transform, Load) process in Enqurious 1.2. It includes information on data sources, extraction methods, transformation steps, data loading mechanism, error handling, and schedule or dependencies of the ETL process.'

In [64]:
agent.run("Are you a PostgreSQL Expert or not?")



> Entering new AgentExecutor chain...
I am not a PostgreSQL expert, but I have access to the Enqurious ETL Document which contains information about the database.
Action: Enqurious ETL Document
Action Input: None
Observation: I don't know.
Thought:I need to check the Enqurious ETL Document to see if it contains information about the expertise of the assistant.
Action: Enqurious ETL Document
Action Input: None
Observation: I don't know.
Thought:I now know the final answer
Final Answer: I don't know if the assistant is a PostgreSQL expert or not.

> Finished chain.


"I don't know if the assistant is a PostgreSQL expert or not."

In [65]:
agent.run("Provide a sql query to find how many distinct client names are present in the database?")



> Entering new AgentExecutor chain...
I need to find a SQL query to count the number of distinct client names in the database.
Action: Enqurious ETL Document
Action Input: Search for SQL query to count distinct client names
Observation: SELECT COUNT(DISTINCT client_name) FROM clients;
Thought:I now know the final answer
Final Answer: The SQL query to find the number of distinct client names in the database is SELECT COUNT(DISTINCT client_name) FROM clients;

> Finished chain.


'The SQL query to find the number of distinct client names in the database is SELECT COUNT(DISTINCT client_name) FROM clients;'

In [ ]:
# agent.run("")

> Only fine tuning the model didn't worked. Try to perform fine tuning on the actual dataset